# FASTTEXT - SPAM ASSASIN

In [1]:
!pip install fasttext

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 2.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-3.0.3-py3-none-any.whl.metadata (10 kB)
Using cached pybind11-3.0.3-py3-none-any.whl (313 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp312-cp312-linux_x86_64.whl size=4653978 sha256=a7dfd53e8a5cc70e7fe306a4e45800659eaf6ae56211876842ad1638baf9e2f8
  Stored in directory: /root/.cache/pip/wheels/20/27/95/a7baf1b435f1cbde017cabdf1e9688526d2b0e929255a359c6
Successfully built fasttext


In [ ]:
import numpy as np
import pandas as pd
import re
import os
import tempfile
import fasttext
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

# Load preprocessed data
print("Loading preprocessed data...")
data = pd.read_csv('../DATASETS/SA_preprocessed.csv')

print(f"Dataset shape: {data.shape}")
print(f"Target distribution:\n{data['label'].value_counts()}")

# Prepare data
X = data['processed_text'].fillna('')
y = data['label'].values

print(f"\nTotal samples: {len(X)}")
print(f"Spam: {sum(y)}, Ham: {len(y) - sum(y)}")

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

Loading preprocessed data...
Dataset shape: (6045, 4)
Target distribution:
label
0    4149
1    1896
Name: count, dtype: int64

Total samples: 6045
Spam: 1896, Ham: 4149

Training samples: 4836
Testing samples: 1209


## FastText Supervised Classification

In [3]:
# additional preprocessing for fasttext
def write_fasttext_file(texts, labels, filepath):
    # fasttext supervised format: __label__<label> <text>
    with open(filepath, 'w', encoding='utf-8') as f:
        for text, label in zip(texts, labels):
            # Map: 0 -> ham, 1 -> spam
            label_str = 'spam' if label == 1 else 'ham'
            f.write(f"__label__{label_str} {text}\n")

# Write train and test files
train_file = 'fasttext_train.txt'
test_file = 'fasttext_test.txt'

write_fasttext_file(X_train.values, y_train, train_file)
write_fasttext_file(X_test.values, y_test, test_file)

print(f"Training file written: {train_file}")
print(f"Test file written: {test_file}")

# Verify format
with open(train_file, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i < 3:
            print(f"Line {i}: {line[:120]}...")
        else:
            break

Training file written: fasttext_train.txt
Test file written: fasttext_test.txt
Line 0: __label__spam cost effective mlm bizop phone screened leads hello premium phone qualified business opportunity leads if ...
Line 1: __label__ham bush orders sharon to obey un resolutions so us can gather support for breaking of un resolutions to punish...
Line 2: __label__ham re ilug mirror a website you could try httrack available here urladdr it does recursive grabs of stuff gets...


In [4]:
# Train fasttext supervised model
print("Training FastText supervised model...")
model = fasttext.train_supervised(
    input=train_file,
    lr=0.1,           # learning rate
    dim=100,           # word vector dimension
    epoch=25,          # number of epochs
    wordNgrams=2,      # use bigrams
    minCount=1,        # min word frequency
    loss='softmax',    # loss function
    verbose=2
)

print(f"\nModel trained.")
print(f"Number of words: {len(model.words)}")
print(f"Number of labels: {len(model.labels)}")
print(f"Labels: {model.labels}")

Training FastText supervised model...

Model trained.
Number of words: 40632
Number of labels: 2
Labels: ['__label__ham', '__label__spam']


In [5]:
# Evaluate on test set
n_samples, precision, recall = model.test(test_file)
print(f"Samples:   {n_samples}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {2 * precision * recall / (precision + recall):.4f}")

# Per-label metrics
print("\nPer-label results:")
label_results = model.test_label(test_file)
for label, metrics in label_results.items():
    print(f"  {label}: Precision={metrics['precision']:.4f}, Recall={metrics['recall']:.4f}, F1={metrics['f1score']:.4f}")

Samples:   1209
Precision: 0.9777
Recall:    0.9777
F1 Score:  0.9777

Per-label results:
  __label__spam: Precision=0.9583, Recall=0.9710, F1=0.9646
  __label__ham: Precision=0.9867, Recall=0.9807, F1=0.9837


## FastText Unsupervised Embeddings + Classifiers

Train unsupervised word embeddings with FastText (skipgram), then use the sentence vectors as features for Naive Bayes and SVM

In [6]:
unsup_file = 'fasttext_unsupervised.txt'

all_texts = pd.concat([X_train, X_test])
with open(unsup_file, 'w', encoding='utf-8') as f:
    for text in all_texts.values:
        clean_text = str(text).replace('\n', ' ').replace('\r', ' ').strip()
        f.write(clean_text + '\n')

print(f"Unsupervised training file written: {unsup_file}")
print(f"Total lines: {len(all_texts)}")

Unsupervised training file written: fasttext_unsupervised.txt
Total lines: 6045


In [7]:
# train unsupervised fasttext model using skipgram
print("Training FastText unsupervised model (skipgram)...")
ft_unsup = fasttext.train_unsupervised(
    input=unsup_file,
    model='skipgram',
    dim=100,           # embedding dimension
    epoch=5,           # number of epochs
    lr=0.05,           # learning rate
    ws=5,              # context window size
    minCount=2,        # minimum word frequency
    minn=3,            # min char n-gram length
    maxn=6,            # max char n-gram length
    verbose=2
)

print(f"\nModel trained.")
print(f"Vocabulary size: {len(ft_unsup.words)}")
print(f"Embedding dimension: {ft_unsup.get_dimension()}")

Training FastText unsupervised model (skipgram)...

Model trained.
Vocabulary size: 27189
Embedding dimension: 100


In [8]:
# Generate sentence vectors for train and test data
print("Generating sentence vectors...")

def get_sentence_vectors(texts, model):
    """Get fasttext sentence vectors for a list of texts."""
    vectors = []
    for text in texts:
        clean_text = str(text).replace('\n', ' ').replace('\r', ' ').strip()
        vec = model.get_sentence_vector(clean_text)
        vectors.append(vec)
    return np.array(vectors)

X_train_ft = get_sentence_vectors(X_train.values, ft_unsup)
X_test_ft = get_sentence_vectors(X_test.values, ft_unsup)

print(f"FastText training vectors shape: {X_train_ft.shape}")
print(f"FastText test vectors shape: {X_test_ft.shape}")

Generating sentence vectors...
FastText training vectors shape: (4836, 100)
FastText test vectors shape: (1209, 100)


In [9]:
# Classifier 2: SVM
print("FastText Embeddings + SVM")

ft_svm = SVC(kernel='rbf', C=1.0, random_state=42)
ft_svm.fit(X_train_ft, y_train)
ft_svm_pred = ft_svm.predict(X_test_ft)

print(f"Accuracy: {accuracy_score(y_test, ft_svm_pred):.4f}")
print(f"F1 Score: {f1_score(y_test, ft_svm_pred):.4f}")
print()
print(classification_report(y_test, ft_svm_pred, target_names=['Ham', 'Spam']))

FastText Embeddings + SVM
Accuracy: 0.9727
F1 Score: 0.9565

              precision    recall  f1-score   support

         Ham       0.98      0.98      0.98       830
        Spam       0.96      0.96      0.96       379

    accuracy                           0.97      1209
   macro avg       0.97      0.97      0.97      1209
weighted avg       0.97      0.97      0.97      1209



In [18]:
# Get predictions from the supervised model
predictions = model.predict(list(X_test.values.astype(str)))
# predictions[0] contains the labels like [['__label__ham'], ['__label__spam'], ...]
pred_labels = [p[0].replace('__label__', '') for p in predictions[0]]

# Create a DataFrame for analysis
results_df = pd.DataFrame({
    'text': X_test.values,
    'actual': ['spam' if y == 1 else 'ham' for y in y_test],
    'predicted': pred_labels
})

# Filter for misclassified samples
misclassified = results_df[results_df['actual'] != results_df['predicted']]

print(f"Total misclassified samples: {len(misclassified)}")
print("\nFirst 10 misclassified samples (Full Text):")
print("-" * 30)

# Loop through first 10 to print full text
for i, row in misclassified.head(10).iterrows():
    print(f"Sample Index: {i}")
    print(f"Actual: {row['actual']} | Predicted: {row['predicted']}")
    print(f"Text: {row['text']}")
    print("-" * 30)

Total misclassified samples: 27

First 10 misclassified samples (Full Text):
------------------------------
Sample Index: 16
Actual: ham | Predicted: spam
Text: yahoo finance rss beta got a stock ticker for which you would like to have an rss news feed help test the beta rss feeds we have put up o yahoo finance take your favorite ticker say yhoo and put this url in your news aggregator
------------------------------
Sample Index: 46
Actual: spam | Predicted: ham
Text: ireland s first great world cup result o connor quickly moved his cursor down the left side of the web page then showing a flash of his silky skills he cut across to the right hand side and clicked on top ten special offers he zipped past the irish chicken breast fillets at number off nipped round the hb selections ice cream also at number off and went on with the goal firmly in sight budweiser number pack ml cans was e number now e number save e number guinness number pack ml cans was e number now e number save e number 

### Analysis of Misclassifications


**Technical/Newsletter Overlap (False Positives):** Samples like the 'RSS news feed' (Index 16) or 'HTTP Guide' (Index 428) were predicted as Spam. FastText likely picked up on 'newsletter', 'urladdr', and 'subscribe' as strong spam signals, even though these were legitimate ham technical updates.
**Promotional Content in Ham (False Positives):** The Ryanair email (Index 352) contains many keywords associated with marketing (e.g., 'free seat sale', 'buy online', 'special offers'). Because its structure mimics a promotional blast, the model struggled to distinguish it from spam.
**Length and Narrative Complexity (False Negatives):** The 'Enenkio Kingdom' message (Index 548) is an extremely long, political/legal rant. Because it lacks the typical 'hard-sell' keywords of spam (like 'viagra' or 'winner') and uses a formal tone, the model failed to recognize it as junk/scam content.
**News Alerts (False Positives):** Short news snippets regarding computer viruses (Index 58) or credit card theft often contain high-risk keywords ('virus', 'credit card', 'stolen'). Without more context, the model flags the topic rather than the intent.